In [ ]:
import pandas as pd
import numpy as np
import re
from datetime import datetime, timedelta
import datetime as dt
from bs4 import BeautifulSoup as bs
import requests
import json
import time
from tqdm import tqdm
import random
import math
import pickle
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")   # 경고 메세지 끄기
from matplotlib import rcParams
from matplotlib.ticker import StrMethodFormatter

In [ ]:
# seaborn으로 시각화
sns.set()
sns.set_palette('twilight')   # 그래프 색상 팔레트 설정하기

In [ ]:
# 한글 깨지는 현상 방지
# rcParams['font.family'] = 'Malgun Gothic'   # Windows OS일 경우
rcParams['font.family'] = 'AppleGothic'   # Mac OS일 경우
rcParams['axes.unicode_minus'] = False   # 한글 폰트 사용 시, 마이너스 글자가 깨지는 현상 방지

In [ ]:
df = pd.read_pickle('./uc_detail_ugcmerged_v3_reqcount_fixed_230423.p')
C_df = pd.read_pickle('./challenge_detail_v3_230311.p')
G_df = pd.read_pickle('./goal_info_v3_FinalCollection_merged_230304.p')
UC_df = pd.read_pickle('./user_challenge_info_v3_FinalCollection_merged_230301.p')
U_df = pd.read_pickle('./user_details_v3_FinalCollection_merged_230303.p')
F_df = pd.read_pickle('./user_ach_feed_v3_FinalCollection_merged_230301.p')

In [ ]:
pd.set_option('display.max_columns', None)  # 모든 칼럼 출력
# pd.set_option('display.max_rows', None)     # 모든 행 출력

## 1차 전처리
---

In [ ]:
# 누락된 칼럼 전처리 (유저-챌린지 레벨 데이터프레임)
df['challenge_registerInfo_depositSum'] = df['challenge_registerInfo_cashDepositSum'] + df['challenge_registerInfo_prizeDepositSum'] + df['challenge_registerInfo_cardDepositSum'] + df['challenge_registerInfo_transferDepositSum'] + df['challenge_registerInfo_corporateDepositSum']
df = pd.merge(df, C_df[['challenge_id','challenge_type']], how='left', on='challenge_id')

# 누락된 칼럼 전처리 (유저-챌린지-피드 레벨 데이터프레임)
F_df = F_df.merge(df[["user_id", "challenge_id", "challenge_startDate", "challenge_endDate", "goal_goalCategory_level1"]], on=['user_id', 'challenge_id'], how='left')

In [ ]:
# 결측치 처리

# 유저-챌린지-피드 레벨 데이터프레임
tmp_merge = F_df.merge(df[['user_id', 'challenge_id', 'uc_tot_ach_rate']], on=['user_id', 'challenge_id'], how='left')
print(f"피드 레벨 달성률(uc_tot_ach_rate) 결측치 비율: {round((tmp_merge['uc_tot_ach_rate'].isna().sum()/tmp_merge.shape[0])*100,2)}%")
F_df = tmp_merge.dropna(subset=["uc_tot_ach_rate"]).reset_index(drop=True)

# 유저-챌린지 레벨 데이터프레임
# 결측치 비율 확인
print(f"유저-챌린지 레벨 달성률(uc_tot_ach_rate) 결측치 비율: {round((df['uc_tot_ach_rate'].isna().sum()/df.shape[0])*100,2)}%")
# 결측치 제거
df = df.dropna(subset=["uc_tot_ach_rate"]).reset_index(drop=True)

In [ ]:
# 분석에 이용하지 않을 칼럼 제거
df.drop(['user_proc_rate', 'uc_proc_rate', 'uc_proc_rate_bin', 'user_proc_rate_bin', 'user_proc_rate_bin_10',
         'uc_req_tot_ach_count_org', 'uc_tot_ach_rate_org', 
         'goal_isGalleryPossible'], axis = 1, inplace = True)

In [ ]:
# datetime으로 형 변환
df['challenge_startDate'] = pd.to_datetime(df['challenge_startDate'],infer_datetime_format=True)
df['challenge_endDate'] = pd.to_datetime(df['challenge_endDate'],infer_datetime_format=True)
df['challenge_endDate'] = df['challenge_endDate'].dt.strftime('%Y-%m-%d')
df['challenge_endDate'] = pd.to_datetime(df['challenge_endDate'],infer_datetime_format=True)

In [ ]:
# 데이터프레임 정렬
df = df.sort_values(by=['user_id', 'challenge_startDate', 'challenge_endDate'], ignore_index=True)

In [ ]:
# 달성률 이상치 처리

# 달성률 기술 통계량 확인
pd.options.display.float_format = '{:.5f}'.format   # 자연상수 사용 X
print('Before Preprocessing =========================')
print(df['uc_tot_ach_rate'].describe())

# 달성률이 100 초과인 값을 100으로 대체
df['uc_tot_ach_rate'] = df['uc_tot_ach_rate'].apply(lambda x: 100 if x > 100 else x)
print('\nAfter Preprocessing =========================')
print(df['uc_tot_ach_rate'].describe())

In [ ]:
# 종속변수 (다음 등록까지 걸리는 일 수) 생성
def calculate_diff(group):
    
    # 날짜 차이를 append할 리스트
    diff_list = []
    
    if len(group)==1:
        diff_list.append(np.nan)
        group['uc_days_nextChallenge'] = diff_list
        
    elif len(group)>1:
        challenge_startDate_list = group['challenge_startDate'].drop_duplicates(keep='first').tolist()
        challenge_startDate_list = challenge_startDate_list[:-1]
        
        for start in challenge_startDate_list:
            # start와 challenge_startDate가 같은 행의 인덱스 리스트 저장
            group_idx = group.index[group['challenge_startDate'] == start].tolist()
            idx_start = group_idx[0]
            idx_end = group_idx[-1]
            # 날짜 차이를 계산할 데이터프레임
            group_df = group.loc[idx_start:idx_end+1]
            for idx in group_idx:
                idx1_endDate = group_df['challenge_endDate'][idx]
                idx2_startDate = group_df['challenge_startDate'].iloc[-1]
                group_diff = (idx2_startDate - idx1_endDate).days
                diff_list.append(group_diff)
        # 마지막 날짜의 경우 계산이 불가능하므로 np.nan 부여
        cnt = len(group) - len(diff_list)
        for c in range(cnt):
            diff_list.append(np.nan)
        # diff_list를 새로운 칼럼으로 생성
        group['uc_days_nextChallenge'] = diff_list
    return group

df2 = df.groupby('user_id', group_keys=False).apply(calculate_diff)
df2

In [ ]:
# 챌린지 기간 파생변수 생성
df2['challenge_period'] = (df2['challenge_endDate'] - df2['challenge_startDate']).dt.days + 1

In [ ]:
# 챌린지 결과 더미변수 생성
result_dummies = pd.get_dummies(df2['relationUserChallenge_result'])
# 열 이름 변경
new_col_names = {
    'NORMAL': 'uc_result_Normal',
    'OVER_EIGHTY_FIVE': 'uc_result_OverEightyFive', 
    'OVER_HUNDRED': 'uc_result_OverHundred',
}
result_dummies.rename(columns=new_col_names, inplace=True)
# 원 데이터셋과 결합
df2 = pd.concat([df2, result_dummies], axis = 1)
df2.head()

In [ ]:
def create_features(group):
    
    # 이전 평균 경험을 append할 리스트
    uc_deposit_list = []
    uc_feeds_list = []
    uc_req_feeds_list = []
    uc_ach_rate_list = []
    challenge_registerCount_list = []
    challenge_registerCountHistory_list = []
    challenge_deposit_list = []
    
    # 챌린지 종료일 리스트
    challenge_endDate_list = group['challenge_endDate'].drop_duplicates(keep='first').tolist()

    for end in challenge_endDate_list:
        # end와 challenge_startDate가 같은 행의 인덱스 리스트 저장
        group_idx = group.index[group['challenge_endDate'] == end].tolist()
        idx_end = group_idx[-1]
        # 데이터프레임 부분 추출
        group_df = group.loc[:idx_end]
        # 누적 평균 구하기
        tmp_uc_deposit_mean = group_df['relationUserChallenge_registerInfo_deposit'].mean()
        tmp_uc_feeds_mean = group_df['uc_tot_ach_count'].mean()
        tmp_uc_req_feeds_mean = group_df['uc_req_tot_ach_count'].mean()
        tmp_uc_ach_rate_mean = group_df['uc_tot_ach_rate'].mean()
        tmp_challenge_registerCount_mean = group_df['challenge_registerInfo_registerCount'].mean()
        tmp_challenge_registerCountHistory_mean = group_df['challenge_recordHistory_totalRegisterUserCount'].mean()
        tmp_challenge_deposit_mean = group_df['challenge_registerInfo_depositSum'].mean()
        
        for i in range(len(group_idx)):
            uc_deposit_list.append(tmp_uc_deposit_mean)
            uc_feeds_list.append(tmp_uc_feeds_mean)
            uc_req_feeds_list.append(tmp_uc_req_feeds_mean)
            uc_ach_rate_list.append(tmp_uc_ach_rate_mean)
            challenge_registerCount_list.append(tmp_challenge_registerCount_mean)
            challenge_registerCountHistory_list.append(tmp_challenge_registerCountHistory_mean)
            challenge_deposit_list.append(tmp_challenge_deposit_mean)
    
    # 데이터프레임에 파생변수로 저장
    group['uc_deposit_cumMean'] = uc_deposit_list
    group['uc_feeds_cumMean'] = uc_feeds_list
    group['uc_req_feeds_cumMean'] = uc_req_feeds_list
    group['uc_ach_rate_cumMean'] = uc_ach_rate_list
    group['challenge_registerCount_cumMean'] = challenge_registerCount_list
    group['challenge_registerCountHistory_cumMean'] = challenge_registerCountHistory_list
    group['challenge_deposit_cumMean'] = challenge_deposit_list
    
    return group

df2 = df2.groupby('user_id', group_keys=False).apply(create_features)
df2

In [ ]:
# datetime으로 형 변환
df2['relationUserChallenge_createdAt'] = pd.to_datetime(df2['relationUserChallenge_createdAt'],infer_datetime_format=True)
df2['relationUserChallenge_createdAt'] = df2['relationUserChallenge_createdAt'].dt.strftime('%Y-%m-%d')
df2['relationUserChallenge_createdAt'] = pd.to_datetime(df2['relationUserChallenge_createdAt'],infer_datetime_format=True)
df2['user_createdAt'] = pd.to_datetime(df2['user_createdAt'],infer_datetime_format=True)
df2['user_createdAt_year'] = df2['user_createdAt'].dt.year
df2['user_createdAt_yearMonth'] = df2['user_createdAt'].dt.strftime('%Y-%m')
df2

In [ ]:
# 동시에 진행 중인 다른 챌린지 개수 세는 함수
def count_overlappingChallenges(group):
    # 파생 변수에 기본 값으로 0 할당
    group['uc_overlappingChallenges_cnt'] = 0
    # 행별로 겹치는 챌린지 개수 세기
    for index, row in group.iterrows():
        overlapping_challenges = group[(group['challenge_startDate'] <= row['challenge_endDate']) &
                                       (group['challenge_endDate'] >= row['challenge_startDate']) &
                                       (group.index != index)]
        group.at[index, 'uc_overlappingChallenges_cnt'] = len(overlapping_challenges)
        
    return group

df2 = df2.groupby('user_id', group_keys=False).apply(count_overlappingChallenges)
df2

In [ ]:
# 동시에 진행 중인 다른 챌린지가 존재하는지 여부를 나타내는 파생변수 생성
df2['uc_overlappingChallenges_exist'] = df2['uc_overlappingChallenges_cnt'].apply(lambda x: 1 if x != 0 else 0)

In [ ]:
# pickle로 저장
df2.to_pickle('(Panel)Challengers_df_v1.0.pkl')

## 2차 전처리
---

In [ ]:
df3 = df2[['user_id', 'challenge_id', 'relationUserChallenge_id', 'relationUserChallenge_createdAt',
           'goal_id', 'goal_title', 'goal_goalCategory_level1', 'goal_goalCategory_level2', 'goal_goalCategory_level3', 'goal_goalCategory_level4', 'goal_goalCategoryId', 
           'goal_maxAchieveCountPerDay', 'goal_dayOrPeriod',
           'user_createdAt', 'user_createdAt_year', 'user_createdAt_yearMonth', 'user_age', 'user_sex', 
           'user_isFollowerOnly', 'user_followee_cnt', 'user_follower_cnt', 
           'challenge_title', 'challenge_type', 'challenge_startDate', 'challenge_endDate', 'challenge_period', 'challenge_minRegisterDeposit', 'challenge_maxRegisterDeposit', 
           'challenge_registerInfo_depositSum', 'challenge_registerInfo_registerCount', 'challenge_recordHistory_totalRegisterUserCount',
           'challenge_deposit_cumMean', 'challenge_registerCount_cumMean', 'challenge_registerCountHistory_cumMean',
           'relationUserChallenge_publicType', 
           'relationUserChallenge_result', 'uc_result_Normal', 'uc_result_OverEightyFive', 'uc_result_OverHundred',
           'relationUserChallenge_registerInfo_deposit', 'uc_deposit_cumMean', 
           'uc_tot_ach_count', 'uc_req_tot_ach_count', 'uc_tot_ach_rate',
           'uc_feeds_cumMean', 'uc_req_feeds_cumMean', 'uc_ach_rate_cumMean',
           'uc_overlappingChallenges_cnt', 'uc_overlappingChallenges_exist',
           'uc_days_nextChallenge']]

In [ ]:
# 열 이름 변경
col_names = {
    'relationUserChallenge_id': 'userChallenge_id',
    'relationUserChallenge_createdAt': 'userChallenge_createdAt', 
    'goal_goalCategory_level1': 'goal_category_level1',
    'goal_goalCategory_level2': 'goal_category_level2',
    'goal_goalCategory_level3': 'goal_category_level3',
    'goal_goalCategory_level4': 'goal_category_level4',
    'goal_goalCategoryId': 'goal_category_id',
    'goal_maxAchieveCountPerDay': 'goal_req_feedsPerDay',
    'challenge_registerInfo_depositSum': 'challenge_deposit',
    'challenge_registerInfo_registerCount': 'challenge_registerCount',
    'challenge_recordHistory_totalRegisterUserCount': 'challenge_registerCountHistory',
    'relationUserChallenge_publicType': 'uc_publicType',
    'relationUserChallenge_result': 'uc_result',
    'relationUserChallenge_registerInfo_deposit': 'uc_deposit',
    'uc_tot_ach_count': 'uc_feeds',
    'uc_req_tot_ach_count': 'uc_req_feeds',
    'uc_tot_ach_rate': 'uc_ach_rate',
}
df3.rename(columns=col_names, inplace=True)

In [ ]:
df3

In [ ]:
# 종속변수 전처리
# 음수 값 대체
df3['uc_days_nextChallenge_noNeg'] = df3['uc_days_nextChallenge'].apply(lambda x: 0 if x < 0 else x)
# 결측치 대체
crawling_date = pd.to_datetime('2023-02-28')
df3['uc_days_nextChallenge_noNaN'] = df3['uc_days_nextChallenge'].fillna((crawling_date - df3['challenge_endDate']).dt.days)
df3['uc_days_nextChallenge_noNeg_noNaN'] = df3['uc_days_nextChallenge_noNeg'].fillna((crawling_date - df3['challenge_endDate']).dt.days)
df3.head()

In [ ]:
# (uc_overlappingChallenges_cnt / 챌린지 기간 파생변수) 생성
df3['uc_overlappingChallengesCnt_perPeriod'] = df3['uc_overlappingChallenges_cnt'] / df3['challenge_period']

In [ ]:
# 동시에 진행 중인 다른 챌린지와 겹치는 날짜 수의 합을 챌린지 기간으로 나누는 함수
def calculate_overlappingDays(group):
    
    # 각 행의 sumOverlappingDays_perPeriod 값을 append할 리스트
    sumOverlappingDays_perPeriod_list = []
    
    for i in range(len(group)):
        idx = group.iloc[i].name
        start_date = group['challenge_startDate'].iloc[i]
        end_date = group['challenge_endDate'].iloc[i]
        challenge_period = group['challenge_period'].iloc[i]
        
        # 다른 챌린지와 챌린지 기간이 겹치는 날짜 수 총합 계산
        total_overlap_days = 0
        for index, row in group.iterrows():
            if index != idx:  # 0번째 행 제외
                overlap_start = max(start_date, row['challenge_startDate'])
                overlap_end = min(end_date, row['challenge_endDate'])
                overlap_days = max((overlap_end - overlap_start).days + 1, 0)  # 겹치는 날짜 수 계산
                total_overlap_days += overlap_days
        
        # 다른 챌린지와 챌린지 기간이 겹치는 날짜 수 총합 / 챌린지 기간
        sumOverlappingDays_perPeriod = total_overlap_days / challenge_period
        sumOverlappingDays_perPeriod_list.append(sumOverlappingDays_perPeriod)
        
    # sumOverlappingDays_perPeriod_list를 새로운 칼럼으로 생성
    group['uc_sumOverlappingDays_perPeriod'] = sumOverlappingDays_perPeriod_list
    
    return group

df3 = df3.groupby('user_id', group_keys=False).apply(calculate_overlappingDays)
df3

In [ ]:
df3 = df3[['user_id', 'challenge_id', 'userChallenge_id',
           'userChallenge_createdAt', 'goal_id', 'goal_title',
           'goal_category_level1', 'goal_category_level2', 'goal_category_level3',
           'goal_category_level4', 'goal_category_id', 'goal_req_feedsPerDay',
           'goal_dayOrPeriod', 'user_createdAt', 'user_createdAt_year',
           'user_createdAt_yearMonth', 'user_age', 'user_sex',
           'user_isFollowerOnly', 'user_followee_cnt', 'user_follower_cnt',
           'challenge_title', 'challenge_type', 'challenge_startDate',
           'challenge_endDate', 'challenge_period', 'challenge_minRegisterDeposit',
           'challenge_maxRegisterDeposit', 'challenge_deposit',
           'challenge_registerCount', 'challenge_registerCountHistory',
           'challenge_deposit_cumMean', 'challenge_registerCount_cumMean',
           'challenge_registerCountHistory_cumMean', 'uc_publicType', 'uc_result',
           'uc_result_Normal', 'uc_result_OverEightyFive', 'uc_result_OverHundred',
           'uc_deposit', 'uc_deposit_cumMean', 'uc_feeds', 'uc_req_feeds',
           'uc_ach_rate', 'uc_feeds_cumMean', 'uc_req_feeds_cumMean', 'uc_ach_rate_cumMean',
           'uc_overlappingChallenges_cnt', 'uc_overlappingChallengesCnt_perPeriod', 'uc_sumOverlappingDays_perPeriod', 'uc_overlappingChallenges_exist',
           'uc_days_nextChallenge', 'uc_days_nextChallenge_noNeg', 'uc_days_nextChallenge_noNaN', 'uc_days_nextChallenge_noNeg_noNaN']]

In [ ]:
def create_features21(group):
    
    # 이전 평균 경험을 append할 리스트
    uc_feeds_list = []
    uc_req_feeds_list = []
    uc_ach_rate_list = []
    
    # 챌린지 종료일 리스트
    challenge_endDate_list = group['challenge_endDate'].drop_duplicates(keep='first').tolist()
    len_challenge_endDate_list = len(challenge_endDate_list)
    
    if len_challenge_endDate_list==1:
        group_idx = group.index[group['challenge_endDate'] == challenge_endDate_list[0]].tolist()
        for n in range(len(group_idx)):
            uc_feeds_list.append(np.nan)
            uc_req_feeds_list.append(np.nan)
            uc_ach_rate_list.append(np.nan)
        
    else:
        group_idx = group.index[group['challenge_endDate'] == challenge_endDate_list[0]].tolist()
        for k in range(len(group_idx)):
            uc_feeds_list.append(np.nan)
            uc_req_feeds_list.append(np.nan)
            uc_ach_rate_list.append(np.nan)

        for end in challenge_endDate_list[1:]:
            # end와 challenge_endDate 같은 행의 인덱스 리스트 저장
            group_idx = group.index[group['challenge_endDate'] == end].tolist()
            len_group_idx = len(group_idx)
            idx_end = group_idx[-1]
            # 데이터프레임 부분 추출
            group_df = group.loc[:idx_end]
            group_df = group_df.iloc[:-len_group_idx]
            # 누적 평균 구하기
            tmp_uc_feeds_mean = group_df['uc_feeds'].mean()
            tmp_uc_req_feeds_mean = group_df['uc_req_feeds'].mean()
            tmp_uc_ach_rate_mean = group_df['uc_ach_rate'].mean()

            for i in range(len(group_idx)):
                uc_feeds_list.append(tmp_uc_feeds_mean)
                uc_req_feeds_list.append(tmp_uc_req_feeds_mean)
                uc_ach_rate_list.append(tmp_uc_ach_rate_mean)
    
    # 데이터프레임에 파생변수로 저장
    group['uc_feeds_cumMean_before'] = uc_feeds_list
    group['uc_req_feeds_cumMean_before'] = uc_req_feeds_list
    group['uc_ach_rate_cumMean_before'] = uc_ach_rate_list
    
    return group

df3 = df3.groupby('user_id', group_keys=False).apply(create_features21)
df3

In [ ]:
def create_features22(group):
    
    # 이전 평균 경험을 append할 리스트
    uc_deposit_list = []
    challenge_registerCount_list = []
    challenge_registerCountHistory_list = []
    challenge_deposit_list = []
    
    # 챌린지 종료일 리스트
    challenge_startDate_list = group['challenge_startDate'].drop_duplicates(keep='first').tolist()
    len_challenge_startDate = len(challenge_startDate_list)
    
    if len_challenge_startDate==1:
        group_idx = group.index[group['challenge_startDate'] == challenge_startDate_list[0]].tolist()
        for n in range(len(group_idx)):
            uc_deposit_list.append(np.nan)
            challenge_registerCount_list.append(np.nan)
            challenge_registerCountHistory_list.append(np.nan)
            challenge_deposit_list.append(np.nan)
        
    else:
        group_idx = group.index[group['challenge_startDate'] == challenge_startDate_list[0]].tolist()
        for k in range(len(group_idx)):
            uc_deposit_list.append(np.nan)
            challenge_registerCount_list.append(np.nan)
            challenge_registerCountHistory_list.append(np.nan)
            challenge_deposit_list.append(np.nan)

        for start in challenge_startDate_list[1:]:
            # start와 challenge_startDate 같은 행의 인덱스 리스트 저장
            group_idx = group.index[group['challenge_startDate'] == start].tolist()
            len_group_idx = len(group_idx)
            idx_end = group_idx[-1]
            # 데이터프레임 부분 추출
            group_df = group.loc[:idx_end]
            group_df = group_df.iloc[:-len_group_idx]
            # 누적 평균 구하기
            tmp_uc_deposit_mean = group_df['uc_deposit'].mean()
            tmp_challenge_registerCount_mean = group_df['challenge_registerCount'].mean()
            tmp_challenge_registerCountHistory_mean = group_df['challenge_registerCountHistory'].mean()
            tmp_challenge_deposit_mean = group_df['challenge_deposit'].mean()

            for i in range(len(group_idx)):
                uc_deposit_list.append(tmp_uc_deposit_mean)
                challenge_registerCount_list.append(tmp_challenge_registerCount_mean)
                challenge_registerCountHistory_list.append(tmp_challenge_registerCountHistory_mean)
                challenge_deposit_list.append(tmp_challenge_deposit_mean)
    
    # 데이터프레임에 파생변수로 저장
    group['uc_deposit_cumMean_before'] = uc_deposit_list
    group['challenge_registerCount_cumMean_before'] = challenge_registerCount_list
    group['challenge_registerCountHistory_cumMean_before'] = challenge_registerCountHistory_list
    group['challenge_deposit_cumMean_before'] = challenge_deposit_list
    
    return group

df3 = df3.groupby('user_id', group_keys=False).apply(create_features22)
df3

In [ ]:
# challenge_deposit
df3['challenge_deposit_diff'] = np.where(df3['challenge_deposit_cumMean_before'].isnull(),
                                         0,
                                         df3['challenge_deposit'] - df3['challenge_deposit_cumMean_before'])

# challenge_registerCount
df3['challenge_registerCount_diff'] = np.where(df3['challenge_registerCount_cumMean_before'].isnull(),
                                               0,
                                               df3['challenge_registerCount'] - df3['challenge_registerCount_cumMean_before'])

# challenge_registerCountHistory
df3['challenge_registerCountHistory_diff'] = np.where(df3['challenge_registerCountHistory_cumMean_before'].isnull(),
                                                      0,
                                                      df3['challenge_registerCountHistory'] - df3['challenge_registerCountHistory_cumMean_before'])

# uc_deposit
df3['uc_deposit_diff'] = np.where(df3['uc_deposit_cumMean_before'].isnull(),
                                  0,
                                  df3['uc_deposit'] - df3['uc_deposit_cumMean_before'])

# uc_feeds
df3['uc_feeds_diff'] = np.where(df3['uc_feeds_cumMean_before'].isnull(),
                                0,
                                df3['uc_feeds'] - df3['uc_feeds_cumMean_before'])

# uc_req_feeds
df3['uc_req_feeds_diff'] = np.where(df3['uc_req_feeds_cumMean_before'].isnull(),
                                    0,
                                    df3['uc_req_feeds'] - df3['uc_req_feeds_cumMean_before'])

# uc_ach_rate
df3['uc_ach_rate_diff'] = np.where(df3['uc_ach_rate_cumMean_before'].isnull(),
                                   0,
                                   df3['uc_ach_rate'] - df3['uc_ach_rate_cumMean_before'])

In [ ]:
df3 = df3[['user_id', 'challenge_id', 'userChallenge_id',
           'userChallenge_createdAt', 'goal_id', 'goal_title',
           'goal_category_level1', 'goal_category_level2', 'goal_category_level3',
           'goal_category_level4', 'goal_category_id', 'goal_req_feedsPerDay',
           'goal_dayOrPeriod', 'user_createdAt', 'user_createdAt_year',
           'user_createdAt_yearMonth', 'user_age', 'user_sex',
           'user_isFollowerOnly', 'user_followee_cnt', 'user_follower_cnt',
           'challenge_title', 'challenge_type', 'challenge_startDate',
           'challenge_endDate', 'challenge_period', 'challenge_minRegisterDeposit',
           'challenge_maxRegisterDeposit', 'challenge_deposit',
           'challenge_registerCount', 'challenge_registerCountHistory',
           'challenge_deposit_cumMean', 'challenge_registerCount_cumMean',
           'challenge_registerCountHistory_cumMean', 'challenge_deposit_cumMean_before', 
           'challenge_registerCount_cumMean_before', 'challenge_registerCountHistory_cumMean_before', 
           'challenge_deposit_diff', 'challenge_registerCount_diff', 'challenge_registerCountHistory_diff',
           'uc_publicType', 'uc_result',
           'uc_result_Normal', 'uc_result_OverEightyFive', 'uc_result_OverHundred',
           'uc_deposit', 'uc_deposit_cumMean', 'uc_deposit_cumMean_before', 'uc_deposit_diff',
           'uc_feeds', 'uc_req_feeds', 'uc_ach_rate', 
           'uc_feeds_cumMean', 'uc_req_feeds_cumMean', 'uc_ach_rate_cumMean',
           'uc_feeds_cumMean_before', 'uc_req_feeds_cumMean_before', 'uc_ach_rate_cumMean_before',
           'uc_feeds_diff', 'uc_req_feeds_diff', 'uc_ach_rate_diff',
           'uc_overlappingChallenges_cnt', 'uc_overlappingChallengesCnt_perPeriod', 'uc_sumOverlappingDays_perPeriod', 'uc_overlappingChallenges_exist',
           'uc_days_nextChallenge', 'uc_days_nextChallenge_noNeg', 'uc_days_nextChallenge_noNaN', 'uc_days_nextChallenge_noNeg_noNaN']]

In [ ]:
df3['challenge_startDate_year'] = df3['challenge_startDate'].dt.year
df3['challenge_startDate_yearMonth'] = df3['challenge_startDate'].dt.strftime('%Y-%m')
df3['challenge_endDate_year'] = df3['challenge_endDate'].dt.year
df3['challenge_endDate_yearMonth'] = df3['challenge_endDate'].dt.strftime('%Y-%m')

In [ ]:
df3 = df3[['user_id', 'challenge_id', 'userChallenge_id',
           'userChallenge_createdAt', 'goal_id', 'goal_title',
           'goal_category_level1', 'goal_category_level2', 'goal_category_level3',
           'goal_category_level4', 'goal_category_id', 'goal_req_feedsPerDay',
           'goal_dayOrPeriod', 'user_createdAt', 'user_createdAt_year',
           'user_createdAt_yearMonth', 'user_age', 'user_sex',
           'user_isFollowerOnly', 'user_followee_cnt', 'user_follower_cnt',
           'challenge_title', 'challenge_type', 
           'challenge_startDate', 'challenge_startDate_year', 'challenge_startDate_yearMonth',
           'challenge_endDate', 'challenge_endDate_year', 'challenge_endDate_yearMonth',
           'challenge_period', 'challenge_minRegisterDeposit',
           'challenge_maxRegisterDeposit', 'challenge_deposit',
           'challenge_registerCount', 'challenge_registerCountHistory',
           'challenge_deposit_cumMean', 'challenge_registerCount_cumMean',
           'challenge_registerCountHistory_cumMean', 'challenge_deposit_cumMean_before', 
           'challenge_registerCount_cumMean_before', 'challenge_registerCountHistory_cumMean_before', 
           'challenge_deposit_diff', 'challenge_registerCount_diff', 'challenge_registerCountHistory_diff',
           'uc_publicType', 'uc_result',
           'uc_result_Normal', 'uc_result_OverEightyFive', 'uc_result_OverHundred',
           'uc_deposit', 'uc_deposit_cumMean', 'uc_deposit_cumMean_before', 'uc_deposit_diff',
           'uc_feeds', 'uc_req_feeds', 'uc_ach_rate', 
           'uc_feeds_cumMean', 'uc_req_feeds_cumMean', 'uc_ach_rate_cumMean',
           'uc_feeds_cumMean_before', 'uc_req_feeds_cumMean_before', 'uc_ach_rate_cumMean_before',
           'uc_feeds_diff', 'uc_req_feeds_diff', 'uc_ach_rate_diff',
           'uc_overlappingChallenges_cnt', 'uc_overlappingChallengesCnt_perPeriod', 'uc_sumOverlappingDays_perPeriod', 'uc_overlappingChallenges_exist',
           'uc_days_nextChallenge', 'uc_days_nextChallenge_noNeg', 'uc_days_nextChallenge_noNaN', 'uc_days_nextChallenge_noNeg_noNaN']]

In [ ]:
# pickle로 저장
df3.to_pickle('(Panel)Challengers_df_v2.pkl')

---

In [ ]:
df3['uc_ach_rate_100'] = np.where(df3['uc_ach_rate'] >= 100, 100, df3['uc_ach_rate'])

In [ ]:
def create_features(group):
    
    # 이전 평균 경험을 append할 리스트
    uc_ach_rate_list = []
    
    # 챌린지 종료일 리스트
    challenge_endDate_list = group['challenge_endDate'].drop_duplicates(keep='first').tolist()

    for end in challenge_endDate_list:
        # end와 challenge_startDate가 같은 행의 인덱스 리스트 저장
        group_idx = group.index[group['challenge_endDate'] == end].tolist()
        idx_end = group_idx[-1]
        # 데이터프레임 부분 추출
        group_df = group.loc[:idx_end]
        # 누적 평균 구하기
        tmp_uc_ach_rate_mean = group_df['uc_ach_rate_100'].mean()
        
        for i in range(len(group_idx)):
            uc_ach_rate_list.append(tmp_uc_ach_rate_mean)
    
    # 데이터프레임에 파생변수로 저장
    group['uc_ach_rate_cumMean_100'] = uc_ach_rate_list
    
    return group

df3 = df3.groupby('user_id', group_keys=False).apply(create_features)
df3

In [ ]:
def create_features21(group):
    
    # 이전 평균 경험을 append할 리스트
    uc_ach_rate_list = []
    
    # 챌린지 종료일 리스트
    challenge_endDate_list = group['challenge_endDate'].drop_duplicates(keep='first').tolist()
    len_challenge_endDate_list = len(challenge_endDate_list)
    
    if len_challenge_endDate_list==1:
        group_idx = group.index[group['challenge_endDate'] == challenge_endDate_list[0]].tolist()
        for n in range(len(group_idx)):
            uc_ach_rate_list.append(np.nan)
        
    else:
        group_idx = group.index[group['challenge_endDate'] == challenge_endDate_list[0]].tolist()
        for k in range(len(group_idx)):
            uc_ach_rate_list.append(np.nan)

        for end in challenge_endDate_list[1:]:
            # end와 challenge_endDate 같은 행의 인덱스 리스트 저장
            group_idx = group.index[group['challenge_endDate'] == end].tolist()
            len_group_idx = len(group_idx)
            idx_end = group_idx[-1]
            # 데이터프레임 부분 추출
            group_df = group.loc[:idx_end]
            group_df = group_df.iloc[:-len_group_idx]
            # 누적 평균 구하기
            tmp_uc_ach_rate_mean = group_df['uc_ach_rate_100'].mean()

            for i in range(len(group_idx)):
                uc_ach_rate_list.append(tmp_uc_ach_rate_mean)
    
    # 데이터프레임에 파생변수로 저장
    group['uc_ach_rate_cumMean_before_100'] = uc_ach_rate_list
    
    return group

df3 = df3.groupby('user_id', group_keys=False).apply(create_features21)
df3

In [ ]:
# uc_ach_rate
df3['uc_ach_rate_diff_100'] = np.where(df3['uc_ach_rate_cumMean_before_100'].isnull(),
                                       0,
                                       df3['uc_ach_rate_100'] - df3['uc_ach_rate_cumMean_before_100'])
df3['uc_ach_rate_diff2_100'] = np.where(df3['uc_ach_rate_cumMean_before_100'].isnull(),
                                       np.nan,
                                       df3['uc_ach_rate_100'] - df3['uc_ach_rate_cumMean_before_100'])

---

In [ ]:
# dta 저장 시 에러가 날 가능성이 있는 칼럼 확인
string_columns = df3.select_dtypes(include=[object]).columns
df3[string_columns]

In [ ]:
cols_drop = ['goal_title', 'goal_category_level4', 'challenge_title']
df3 = df3.drop(cols_drop, axis=1)

In [ ]:
# dta 파일로 저장
df3.to_stata('Panel_Challengers_df_v3.dta', write_index=False)